In [0]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("MailIdAndNumberDF").getOrCreate()

# Define data as a list of tuples (mail_id, number)
data = [
    ("test1@example.com", 101),
    ("test2@example.com", 109982),
    ("user@domain.org", 103),
]

# Define column names
columns = ["mail_id", "number"]

# Create DataFrame
df = spark.createDataFrame(data, schema=columns)

# Display the DataFrame
df.show()
df.printSchema()


In [0]:
from pyspark.sql.functions import split, col, expr
splitdf= (
    df.withColumn("mail_id_n", split(col("mail_id"), "@")[0])
      .withColumn("domain", expr("substring_index(mail_id, '@', -1)"))
)
splitdf.show()

In [0]:
splitdf1=(splitdf
    .withColumn("first_2", expr("substring(mail_id_n, 1, 1)"))
      .withColumn("last_2", expr("substring(mail_id_n, -2)"))
      .withColumn("length_3", expr("length(mail_id_n) - 3"))
)
splitdf1.show()

In [0]:
final_df=splitdf1.withColumn("final_mail",expr("concat(first_2, repeat('*',length_3), last_2,'@', domain " ))
final_df.show()

In [0]:
# check data in json / multiline
df = spark.read.format("json").load("/Workspace/Users/pavanikrishnamy28@gmail.com/Databricks_dataeng/data.json")
df.show()
df.printSchema()

In [0]:
# check data in json / multiline
df = spark.read.format("json").option("multiline","true").load("/Workspace/Users/pavanikrishnamy28@gmail.com/Databricks_dataeng/data.json")
df.show()
df.printSchema()

In [0]:
# for struck add .
structdf=df.select(
    "id",
    "name",
    "address.permanent",
    "address.temporary"
)
structdf.show()

In [0]:
#withcolumn will only check for the existing column in the first hierarchy
withdf=(
    df.withColumn("permanent",expr("address.permanent"))
      .withColumn("temporary",expr("address.temporary"))
)
withdf.show()
withdf.printSchema()



In [0]:
#Arrays
from pyspark.sql.functions import explode
arrdf1=df.selectExpr(
    "id",
    "name",
    "address.permanent",
    "address.temporary",
    "explode(products)"
)
arrdf1.show()
arrdf1.printSchema()

In [0]:
cjsondf=spark.read.format("json").option("multiline","true").load("/Workspace/Users/pavanikrishnamy28@gmail.com/Databricks_dataeng/complex_sample_data.json")
cjsondf.show()
cjsondf.printSchema()
cjsondf.count()


In [0]:
explodedf=cjsondf.selectExpr(
    "explode(products) as Product",
    "phone_numbers",
    "explode(orders) as order",
    "name",
    "metadata",
    "id",
    "email",
    "age",
    "active",
    "address"
)
explodedf.show()
explodedf.printSchema()
explodedf.count()

In [0]:
from pyspark.sql.functions import explode,expr
datadf=(
    cjsondf.withColumn("Permanent",expr("address.permanent"))
      .withColumn("Temporary",expr("address.temporary"))
      .withColumn("Product",explode("products"))
      .withColumn("Order",explode("orders"))
      .drop("address","products","orders")
)
datadf.show()
datadf.printSchema()
datadf.count()

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import col, explode_outer
from pyspark.sql.types import StructType, ArrayType

def flatten_all(df: DataFrame) -> DataFrame:
    while True:
        schema = df.schema
        
        # 1. Prioritize flattening StructType fields using withColumn
        struct_fields = [
            field.name for field in schema.fields 
            if isinstance(field.dataType, StructType)
        ]
        
        if struct_fields:
            col_name = struct_fields[0]
            struct_type = schema[col_name].dataType
            
            for nested_field in struct_type.fields:
                nested_name = nested_field.name
                new_col_name = f"{col_name}_{nested_name}"
                df = df.withColumn(new_col_name, col(f"{col_name}.{nested_name}"))
                
            df = df.drop(col_name)
            continue  # Restart loop to process newly created columns
            
        # 2. Handle ArrayType fields using explode_outer
        array_fields = [
            field.name for field in schema.fields 
            if isinstance(field.dataType, ArrayType)
        ]
        
        if array_fields:
            col_name = array_fields[0]
            # Explode array (turns array of structs into a struct, or primitives into flat rows)
            df = df.withColumn(col_name, explode_outer(col(col_name)))
            continue  # Restart loop so any newly exposed structs get flattened
            
        # Exit loop when no structs or arrays remain in the root schema
        break
        
    return df

# Usage:
fully_flattened_df = flatten_all(cjsondf)
fully_flattened_df.printSchema()

In [0]:
import requests, json
from pyspark.sql.functions import from_json, schema_of_json, col, lit
from collections import defaultdict

# Fetch users from public API (nested structs: address → geo, company)
users = requests.get("https://jsonplaceholder.typicode.com/users").json()

# Fetch posts and attach as array of structs per user
posts = requests.get("https://jsonplaceholder.typicode.com/posts").json()
posts_by_user = defaultdict(list)
for p in posts:
    posts_by_user[p["userId"]].append(p)
for u in users:
    u["posts"] = posts_by_user.get(u["id"], [])

# Build Spark DataFrame from JSON strings (Spark Connect compatible — no RDD needed)
json_strings = [json.dumps(u) for u in users]
raw_df = spark.createDataFrame([(s,) for s in json_strings], ["raw_json"])
schema_ddl = raw_df.select(schema_of_json(lit(json_strings[0])).alias("s")).first()["s"]
api_df = raw_df.select(from_json(col("raw_json"), schema_ddl).alias("data")).select("data.*")

api_df.printSchema()
display(api_df)

In [0]:
import requests, json
from pyspark.sql.functions import from_json, schema_of_json, col, lit
from collections import defaultdict

# Fetch users from public API (nested structs: address → geo, company)
users = requests.get("https://jsonplaceholder.typicode.com/users").json()

# Fetch posts and attach as array of structs per user
posts = requests.get("https://jsonplaceholder.typicode.com/posts").json()
posts_by_user = defaultdict(list)
for p in posts:
    posts_by_user[p["userId"]].append(p)
    


for u in users:
   #print(u)
   u["posts"] = posts_by_user.get(u["id"], [])

